<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista2_do_Projeto_de_Sistemas_de_Controle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
import numpy as np
from numpy.polynomial import Polynomial
import re

def coef_amort(Mp_percent):
  # calcula o coeficiente de amortecimento a partir do overshoot em porcentagem
  log_mp = np.log(Mp_percent/100)
  return log_mp/np.sqrt(np.pi**2 + log_mp**2)

def freq_natural(ksi, tss, tss_percent):
  # calcula a frequência natural a partir do ksi, tss e a acomodação
  if(tss_percent == 5):
    constante = 3
  elif(tss_percent == 2):
    constante = 4
  return constante/(tss*ksi)

def sigma_angulos(s, pontos):
  # calcula o somatorio dos angulos entre um ponto s e uma lista de pontos
  sigma = 0
  for i in range(len(pontos)):
    sigma += np.rad2deg(np.atan2((s - pontos[i]).imag, (s - pontos[i]).real))
  return sigma

def phi_modulos(s, pontos):
  # calcula o produtorio dos modulos das distancias entre um ponto s e uma lista de pontos
  phi = 1
  for i in range(len(pontos)):
    phi *= np.abs((s - pontos[i]))
  return phi

def calculo_polos_dominante(s):
  # cálculo os polos dominantes do sistema usando parâmetros especificados
  if(s == '1'):
    print("Overshoot(%): ", end=''); Mp_percent = float(input())
    print("Tempo de acomodação(s): ", end=''); tss = float(input())
    print("Tolerância de acomodação(%): ", end=''); tss_percent = float(input())

    ksi = np.abs(coef_amort(Mp_percent)); ksi = np.round(ksi,4)
    wn = freq_natural(ksi,tss,tss_percent); wn = np.round(wn,4)
    print("Coeficiente de amortecimento: " + str(ksi))
    print("Frequência natural: " + str(wn) + " rad/s")

    wd = wn*np.sqrt(1-np.power(ksi,2)); wd = np.round(wd,4)
    print("Frequência amortecida: " + str(wd))

    s1 = np.round(complex(-ksi*wn,wd), 4); s2 = np.round(complex(-ksi*wn,-wd), 4)
  elif(s == '2'):
    print("Coeficiente de amortecimento: ", end = ''); ksi = float(input())
    print("Frequência natural: ", end = ''); wn = float(input())

    wd = wn*np.sqrt(1-np.power(ksi,2)); wd = np.round(wd,4)
    print("Frequência amortecida: " + str(wd))

    s1 = np.round(complex(-ksi*wn,wd), 4); s2 = np.round(complex(-ksi*wn,-wd), 4)
  elif(s == '3'):
    print("Parte real do polo: ", end=''); real = float(input())
    print("Parte imaginária do polo: ", end=''); imag = float(input())

    s1 = np.round(complex(real,imag), 4);
    s2 = np.round(complex(real,-imag), 4)

  return s1,s2

def projeto_controlador(s, s1, s2, g_num, g_den, h_num, h_den):
  # calcula os zeros e ganho do controlador especificado assim como o Kp, Kd e Ki
  if(s == '1'):
    # controlador PD
    polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()
    phiZ = sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180
    z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
    zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

    kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
    kc = kt/((g_num.coef[-1]*h_num.coef[-1])/(g_den.coef[-1]*h_den.coef[-1]))
    print("Ganho do controlador projetado: " + str(round(kc, 4)))

    kd = kc; kp = kd*z
    print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
    print("Ganho derivativo do controlador: " + str(round(kd, 4)))

  elif(s == '2'):
    # controlador PI
    polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()
    polos = np.append(polos, 0)

    phiZ = sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180
    z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
    zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

    kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
    kc = kt/((g_num.coef[-1]*h_num.coef[-1])/(g_den.coef[-1]*h_den.coef[-1]))
    print("Ganho do controlador projetado: " + str(round(kc, 4)))

    kp = kc; ki = kp*z
    print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
    print("Ganho integrativo do controlador: " + str(round(ki, 4)))

  elif(s == '3'):
    # controlador PID
    polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()
    polos = np.append(polos, 0)

    phiZ = (sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180)/2
    z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
    zeros = np.append(zeros, [-z, -z]); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

    kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
    kc = kt/((g_num.coef[-1]*h_num.coef[-1])/(g_den.coef[-1]*h_den.coef[-1]))
    print("Ganho do controlador projetado: " + str(round(kc, 4)))

    kd = kc; kp = 2*z*kd; ki = kd*np.power(z,2);
    print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
    print("Ganho integrativo do controlador: " + str(round(ki, 4)))
    print("Ganho derivativo do controlador: " + str(round(kd, 4)))

  return kp, ki, kd

In [32]:
num = Polynomial([1, 1]); den = Polynomial([1, 2, 1])
print("G(s) = " + str(num) + '/' + str(den))

print("Digite 1 para discretização pelo método de Euler;")
print("Digite 2 para discretização pelo método de Backward; ou")
print("Digite 3 para discretização pelo método de Tustin.")
s = input()

print("Qual o tempo de amostragem?"); T = (float)(input())
zM = Polynomial([-1, 1]); den_discreto = 0; num_discreto = 0;

# Discretização
if(s == '1'):
  # Método de Euler ou Forward
  # s = (z-1)/T
  for i in range(0,den.degree()+1):
    den_discreto += (T**(den.degree() - i))*den.coef[i]*(zM**i)
  for j in range(0,num.degree()+1):
    num_discreto += (T**(den.degree() - j))*num.coef[j]*(zM**j)
elif(s == '2'):
  # Método Backward
  # s = (z-1)/Tz
  for i in range(0,den.degree()+1):
    den_discreto += ((Polynomial([0, T]))**(den.degree() - i))*den.coef[i]*(zM**i)
  for j in range(0,num.degree()+1):
    num_discreto += ((Polynomial([0, T]))**(den.degree() - j))*num.coef[j]*(zM**j)
elif(s == '3'):
  # Método de Tustin, Bilinear ou Trapezoidal
  # s = (2/T)(z-1)/(z+1)
  zP = Polynomial([1, 1]);
  for i in range(0,den.degree()+1):
    den_discreto += (((T/2)*zP)**(den.degree() - i))*den.coef[i]*(zM**i)
  for j in range(0,num.degree()+1):
    num_discreto += (((T/2)*zP)**(den.degree() - j))*num.coef[j]*(zM**j)

print("G(z) = " + str(num_discreto) + '/' + str(den_discreto))

G(s) = 1.0 + 1.0·x/1.0 + 2.0·x + 1.0·x²
Digite 1 para discretização pelo método de Euler;
Digite 2 para discretização pelo método de Backward; ou
Digite 3 para discretização pelo método de Tustin.
3
Qual o tempo de amostragem?
4
G(z) = 2.0 + 8.0·x + 6.0·x²/1.0 + 6.0·x + 9.0·x²


In [51]:
# Função G(s)
print("Coeficientes do numerador de G(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
g_num = Polynomial(np.flip(coef_arr))
print("Coeficientes do denominador de G(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
g_den = Polynomial(np.flip(coef_arr))

# Função H(s)
print("Coeficientes do numerador de H(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
h_num = Polynomial(np.flip(coef_arr))
print("Coeficientes do denominador de H(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
h_den = Polynomial(np.flip(coef_arr))

# Impressão das funções
print("\nG(x) = (" + str(g_num) + ")/(" + str(g_den) + ")")
print("H(x) = (" + str(h_num) + ")/(" + str(h_den) + ")\n")

# Especificação dos polos dominantes desejados
print("Como deseja informar os polos dominantes?")
print("Digite 1 para Overshoot, Tempo de Acomodação e Tolerância de Acomodação;")
print("Digite 2 para Frequência Natural e Coeficiente de Amortecimento; ou")
print("Digite 3 para informar a parte real e imaginária diretamente.")

s = input(); s_dom = calculo_polos_dominante(s)
s1 = s_dom[0]; s2 = s_dom[1]
print("\ns1 = " + str(s1)); print("s2 = " + str(s2))

# Especificação do controlador
print("\nComo deseja projetar o controlador?")
print("Digite 1 para PD;")
print("Digite 2 para PI; ou")
print("Digite 3 para PID.")

s = input();
kp, ki, kd = projeto_controlador(s, s1, s2, g_num, g_den, h_num, h_den)

# FT do controlador
z1 = -((-kp/kd)+np.sqrt((kp/kd)**2-4*(ki/kd)))/2
z2 = -((-kp/kd)-np.sqrt((kp/kd)**2-4*(ki/kd)))/2
num = kd*Polynomial([z1, 1])*Polynomial([z2, 1]); den = Polynomial([0, 1])
print("\nGc(s) = (" + str(num) + ')/(' + str(den) + ')')

Coeficientes do numerador de G(s): 1
Coeficientes do denominador de G(s): 1 1
Coeficientes do numerador de H(s): 1
Coeficientes do denominador de H(s): 1

G(x) = (1.0)/(1.0 + 1.0·x)
H(x) = (1.0)/(1.0)

Como deseja informar os polos dominantes?
Digite 1 para Overshoot, Tempo de Acomodação e Tolerância de Acomodação;
Digite 2 para Frequência Natural e Coeficiente de Amortecimento; ou
Digite 3 para informar a parte real e imaginária diretamente.
3
Parte real do polo: -1
Parte imaginária do polo: 1

s1 = (-1+1j)
s2 = (-1-1j)

Como deseja projetar o controlador?
Digite 1 para PD;
Digite 2 para PI; ou
Digite 3 para PID.
3
Zero(s) do controlador projetado: 3.4142
Ganho do controlador projetado: 0.2071

Ganho proporcional do controlador: 1.4142
Ganho integrativo do controlador: 2.4142
Ganho derivativo do controlador: 0.2071

Gc(s) = (2.41421356 + 1.41421356·x + 0.20710678·x²)/(0.0 + 1.0·x)
